Architecture search using train/val/test data for **store 1 only** (all items at that store).


In [54]:
# Add project root to path so we can import from data_manipulation and model
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [55]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial

# Imports from data_manipulation and model
from data_manipulation.data_split import create_dataloader, DemandDataset
from model.functions import pinball_loss, rmse, train, get_test_loss

In [56]:
class simple_net(nn.Module):
    def __init__(self, layers:list[int]):
        super().__init__()
        
        self.net = nn.Sequential()

        # Add layers
        for i, layer in enumerate(layers[:-2]):
            self.net.add_module(f"fc{i}", nn.Linear(layer, layers[i+1]))
            if i < len(layers) - 1:
                self.net.add_module(f"relu{i}", nn.ReLU())

        # Add last layer
        self.net.add_module(f"fc{len(layers)-1}", nn.Linear(layers[-2], layers[-1]))
        

    def forward(self, x):
        return self.net(x)

In [57]:
all_specs = [
    "7_day_rolling_ema",	
    "7_day_rolling_mean",	
    "30_day_rolling_ema",	
    "diff_180_day",	
    "diff_90_day",	
    "30_day_rolling_mean",	
    "diff_30_day",	
    "7_day_rolling_min",	
    "7_day_lag",	
    "30_day_rolling_min",	
    "14_day_lag",	
    "diff_365_day",	
]

In [58]:
h_cost = 1
l_cost = 3

num_epochs = 200

In [59]:
in_features = 600
out_features = 50

layer_combos =[
    [in_features, 600, out_features],
    [in_features, 650, out_features],
    [in_features, 500, out_features],
    [in_features, 700, out_features],
    [in_features, 600, 300, out_features],
    [in_features, 600, 200, 100, out_features],
]

In [60]:
from collections import defaultdict

losses = defaultdict(list)

# Number of runs
for i in range(3):
    # get shuffled data
    train_loader, val_loader, test_loader = create_dataloader(
        batch_size=8, 
        test_batch_size=1,
        pin_memory=False,
        data_mask=[("store", 1)],
        specs=all_specs,
        combine_stores=True,
        )

    for layer_combo in layer_combos:
        net_1 = simple_net(layer_combo)
        loss = partial(pinball_loss, h_cost=h_cost, l_cost=l_cost)
        optimizer = torch.optim.Adam(net_1.parameters(), lr=0.001)
        _, _ = train(net_1, optimizer, loss, train_loader, val_loader, epochs=200, eval_interval=10, device="cpu", use_tqdm=False)
        test_loss = get_test_loss(net_1, test_loader, loss, "cpu")
        
        losses[str(layer_combo)].append(torch.tensor(test_loss, dtype=torch.float32).mean().item())

        # Print average loss
        print(f"Test Loss for {str(layer_combo):<50}: {torch.tensor(test_loss, dtype=torch.float32).mean().item():.4f}")


Test Loss for [600, 600, 50]                                    : 10.4014
Test Loss for [600, 650, 50]                                    : 10.4657
Test Loss for [600, 500, 50]                                    : 11.6689
Test Loss for [600, 700, 50]                                    : 10.3682
Test Loss for [600, 600, 300, 50]                               : 11.2782
Test Loss for [600, 600, 200, 100, 50]                          : 11.0483
Test Loss for [600, 600, 50]                                    : 11.0194
Test Loss for [600, 650, 50]                                    : 10.7638
Test Loss for [600, 500, 50]                                    : 10.6586
Test Loss for [600, 700, 50]                                    : 11.0934
Test Loss for [600, 600, 300, 50]                               : 11.0241
Test Loss for [600, 600, 200, 100, 50]                          : 11.7072
Test Loss for [600, 600, 50]                                    : 11.6334
Test Loss for [600, 650, 50]          

In [61]:
# Print the sorted lowest to highest average loss
for layer_combo, losses in sorted(losses.items(), key=lambda x: torch.tensor(x[1]).mean()):
    print(f"Layer Combo: {layer_combo}, Average Loss: {torch.tensor(losses).mean():.4f}")


Layer Combo: [600, 650, 50], Average Loss: 10.6317
Layer Combo: [600, 700, 50], Average Loss: 10.7842
Layer Combo: [600, 500, 50], Average Loss: 10.9155
Layer Combo: [600, 600, 50], Average Loss: 11.0181
Layer Combo: [600, 600, 300, 50], Average Loss: 11.4862
Layer Combo: [600, 600, 200, 100, 50], Average Loss: 11.6390
